In [ ]:
import warnings
warnings.filterwarnings('ignore')
%pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -q confluent-kafka requests matplotlib sseclient-py

In [ ]:
import os
import json
import queue
import threading
import time
import uuid
import io
import base64
import re
from collections import defaultdict

import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sseclient import SSEClient
from confluent_kafka import Consumer, Producer
from IPython.display import HTML, display

pd.set_option('display.max_columns', 40)

# Cấu hình Kiến trúc Kafka & Nguồn cấp dữ liệu (Data Source)
WIKI_SSE_URL = 'https://stream.wikimedia.org/v2/stream/recentchange'
BOOTSTRAP_SERVERS = 'cell-1.streaming.sa-saopaulo-1.oci.oraclecloud.com:9092'
DURATION_SECONDS = 1800
TOPIC = 'DemoStreamingFashion'

SASL_USERNAME = os.environ.get('SASL_USERNAME', '')  # KHONG hardcode
OCI_AUTH_TOKEN = os.environ.get('OCI_AUTH_TOKEN', '')  # KHONG hardcode

RUN_ID = uuid.uuid4().hex[:8]
GROUP_ID = f'wiki_actor_{RUN_ID}'

COMMON_KAFKA_CONF = {
    'bootstrap.servers': BOOTSTRAP_SERVERS,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': SASL_USERNAME,
    'sasl.password': OCI_AUTH_TOKEN,
    'ssl.ca.location': certifi.where(),
}

PRODUCER_CONF = {**COMMON_KAFKA_CONF, 'client.id': f'prod_{RUN_ID}', 'linger.ms': 10, 'acks': '1'}
CONSUMER_CONF = {**COMMON_KAFKA_CONF, 'client.id': f'cons_{RUN_ID}', 'group.id': GROUP_ID, 'auto.offset.reset': 'latest', 'enable.auto.commit': True}

producer_lock = threading.Lock()
producer_stats = {'generated': 0, 'delivered': 0, 'failed': 0, 'error': 'Khởi tạo luồng kết nối dữ liệu máy chủ...'}
local_queue = queue.Queue()
use_fallback = True

# ------------------------------------------------------------------------------
# 1. THUẬT TOÁN PHÂN LỚP NGUỒN GỐC DỮ LIỆU (ACTOR CLASSIFICATION ALGORITHM)
# ------------------------------------------------------------------------------
def classify_actor(event):
    """
    Phân lớp đối tượng phát sinh sự kiện và gán dải màu chuẩn Material Design.
    """
    bot_flag = event.get('bot', False)
    user_id = str(event.get('user', ''))

    # Lớp 1: Tác tử tự động (Automated Scripts/Bots) -> Màu Tím Amethyst
    if bot_flag:
        return "Tác tử tự động", "#8E24AA", "#F3E5F5"

    # Lớp 2: Thực thể không định danh (IP-based) -> Màu Vàng Amber
    if re.match(r'^(\d{1,3}\.){3}\d{1,3}$', user_id) or ':' in user_id:
        return "Thực thể ẩn danh", "#F39C12", "#FFF8E1"

    # Lớp 3: Quản trị hệ thống (System Admin) -> Màu Đỏ Carmine
    if any(kw in user_id.lower() for kw in ['admin', 'sysop', 'script', 'tool', 'bot']):
        return "Tác vụ hệ thống", "#E53935", "#FFEBEE"

    # Lớp 4: Người dùng phổ thông (Registered) -> Màu Xanh Azure
    return "Thực thể định danh", "#1E88E5", "#E3F2FD"

def delivery_report(err, msg):
    with producer_lock:
        if err:
            producer_stats['failed'] += 1
            producer_stats['error'] = f"Ngoại lệ: {str(err)}"
        else:
            producer_stats['delivered'] += 1

def wikimedia_streaming_producer_worker(producer, stop_event):
    global use_fallback
    while not stop_event.is_set():
        try:
            headers = {'Accept': 'text/event-stream', 'User-Agent': 'Academic Research Application / 1.0'}
            response = requests.get(WIKI_SSE_URL, headers=headers, stream=True, timeout=25)
            client = SSEClient(response)

            for msg in client.events():
                if stop_event.is_set(): break
                if not msg.data: continue

                try:
                    event = json.loads(msg.data)
                    # Sàng lọc các sự kiện thay đổi trạng thái nội dung (Edit, New)
                    if event.get('type') in ['edit', 'new', 'log']:
                        title = event.get('title', 'N/A')
                        user = event.get('user', 'N/A')

                        # Thực thi phân lớp
                        actor_type, text_color, bg_color = classify_actor(event)

                        event_payload = {
                            'run_id': RUN_ID,
                            'title': f"Thực thi cập nhật tại: [{title}] bởi định danh: {user}"[:180],
                            'actor': actor_type,
                            't_color': text_color,
                            'b_color': bg_color
                        }

                        payload = json.dumps(event_payload).encode('utf-8')
                        local_queue.put(payload)

                        if not use_fallback:
                            try:
                                producer.produce(TOPIC, value=payload, on_delivery=delivery_report)
                                producer.poll(0)
                            except: pass

                        with producer_lock:
                            producer_stats['generated'] += 1
                            producer_stats['error'] = 'Luồng dữ liệu duy trì tính ổn định'
                except:
                    pass
        except Exception as exc:
            with producer_lock:
                producer_stats['error'] = f"Thiết lập lại kết nối (Lỗi: {str(exc)[:20]})..."
            time.sleep(3)

# ------------------------------------------------------------------------------
# 2. MÔ HÌNH HÓA TRỰC QUAN TRÊN MẶT PHẲNG (2D BAR CHART)
# ------------------------------------------------------------------------------
def generate_2d_bar_chart(rows):
    categories = ['Tác tử\ntự động', 'Thực thể\nẩn danh', 'Tác vụ\nhệ thống', 'Thực thể\nđịnh danh']
    count_keys = ['Tác tử tự động', 'Thực thể ẩn danh', 'Tác vụ hệ thống', 'Thực thể định danh']
    counts = {k: 0 for k in count_keys}

    for row in rows:
        act = row.get('actor', '')
        if 'Tác tử' in act: counts['Tác tử tự động'] += 1
        elif 'ẩn danh' in act: counts['Thực thể ẩn danh'] += 1
        elif 'hệ thống' in act: counts['Tác vụ hệ thống'] += 1
        else: counts['Thực thể định danh'] += 1

    values = [counts[k] for k in count_keys]

    # Dải màu sáng, sang trọng và chuẩn Pro (Material Design)
    colors = ['#8E24AA', '#FFB300', '#E53935', '#1E88E5']

    fig, ax = plt.subplots(figsize=(6.5, 4.5), facecolor='white')

    x_pos = np.arange(len(categories))
    width = 0.55

    # edgecolor='white' và linewidth=2 tạo đường viền sắc nét phân tách các cột với nền
    bars = ax.bar(x_pos, values, width, color=colors, edgecolor='white', linewidth=2, alpha=0.95)

    # Bổ sung nhãn số lượng (Data Labels) ở đỉnh mỗi cột
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 6),  # Dịch lên 6 points để chữ thoáng hơn
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=11, fontweight='900', color='#2c3e50')

    # Tinh chỉnh hiển thị
    ax.set_xticks(x_pos)
    ax.set_xticklabels(categories, fontsize=9, fontweight='bold', color='#34495e')
    ax.set_ylabel('Tần suất phát sinh sự kiện', fontsize=10, fontweight='bold', color='#2c3e50', labelpad=10)
    ax.set_title('Phân Bố Tần Suất Theo Phân Lớp Thực Thể', fontsize=12, fontweight='bold', color='#1a252f', pad=20)

    # Định dạng Gridline tinh tế
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, color='#95a5a6')
    ax.set_axisbelow(True)

    # Loại bỏ các đường viền (spines) không cần thiết để đồ thị phẳng và hiện đại
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_color('#bdc3c7')
    ax.spines['bottom'].set_linewidth(1.5)

    # Xóa các gạch chỉ số trục Y để giao diện sạch 100%
    ax.tick_params(axis='y', length=0)

    plt.subplots_adjust(left=0.1, right=0.95, top=0.85, bottom=0.15)
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    img_str = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return f"data:image/png;base64,{img_str}"

# ------------------------------------------------------------------------------
# 3. KẾT XUẤT BẢNG ĐIỀU KHIỂN (HTML/CSS RENDERING)
# ------------------------------------------------------------------------------
class MockMessage:
    def __init__(self, val): self._val = val
    def value(self): return self._val
    def error(self): return None

def generate_full_dashboard(rows, consumed_count, started_at):
    with producer_lock: pstats = dict(producer_stats)
    elapsed = time.monotonic() - started_at
    mps = consumed_count / elapsed if elapsed > 0 else 0

    status_text = f"TRẠNG THÁI HỆ THỐNG: {pstats['error']} | LƯU LƯỢNG TRUYỀN TẢI: {mps:.1f} evt/s"

    recent_rows = rows[-8:] if len(rows) > 0 else []
    table_rows = ""
    for r in reversed(recent_rows):
        table_rows += f"""
        <tr>
            <td style="font-size: 0.95em; width: 70%; color: #2c3e50; line-height: 1.4;">{r['title']}</td>
            <td style="width: 30%; text-align: center;">
                <span class="badge" style="background-color: {r['b_color']}; color: {r['t_color']}; border: 1px solid {r['t_color']}40;">{r['actor']}</span>
            </td>
        </tr>
        """

    chart_img_src = generate_2d_bar_chart(rows)

    html = f"""
    <style>
        .dashboard-container {{ font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; padding: 25px; background: #fdfdfd; border-radius: 8px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); color: #333333; border: 1px solid #e0e0e0; }}
        .header-title {{ margin-top: 0; color: #1a252f; font-size: 1.4em; font-weight: 700; text-transform: uppercase; border-bottom: 2px solid #2980b9; padding-bottom: 12px; margin-bottom: 15px; letter-spacing: 0.5px; }}
        .status-banner {{ padding: 10px 15px; border-radius: 4px; font-weight: 600; font-size: 0.85em; margin-bottom: 20px; background-color: #e8f4f8; color: #2980b9; border-left: 4px solid #3498db; }}
        .stats-grid {{ display: flex; gap: 15px; margin-bottom: 20px; }}
        .stat-card {{ flex: 1; background: #ffffff; padding: 15px; border-radius: 6px; text-align: center; border: 1px solid #ecf0f1; box-shadow: 0 2px 4px rgba(0,0,0,0.02); }}
        .stat-title {{ font-size: 0.7em; color: #7f8c8d; text-transform: uppercase; font-weight: 700; margin-bottom: 8px; letter-spacing: 0.5px; }}
        .stat-value {{ font-size: 1.5em; font-weight: 700; color: #2c3e50; }}
        .report-grid {{ display: flex; gap: 20px; align-items: stretch; }}
        .report-col {{ flex: 1; background: #ffffff; padding: 20px; border-radius: 6px; border: 1px solid #ecf0f1; box-shadow: 0 2px 4px rgba(0,0,0,0.02); }}
        .report-title {{ font-size: 0.95em; font-weight: 700; color: #34495e; margin-bottom: 15px; text-transform: uppercase; border-bottom: 1px solid #bdc3c7; padding-bottom: 8px; text-align: left; }}
        .data-table {{ width: 100%; border-collapse: collapse; font-size: 0.9em; }}
        .data-table th {{ background-color: #f9fbfb; color: #7f8c8d; padding: 10px; text-align: left; border-bottom: 2px solid #ecf0f1; font-weight: 600; text-transform: uppercase; font-size: 0.85em; }}
        .data-table td {{ padding: 10px 10px; border-bottom: 1px solid #f4f6f7; vertical-align: middle; }}
        .badge {{ padding: 5px 12px; border-radius: 4px; font-weight: 600; display: inline-block; font-size: 0.85em; letter-spacing: 0.3px; box-shadow: 0 1px 2px rgba(0,0,0,0.05); }}
    </style>

    <div class="dashboard-container">
        <h2 class="header-title">HỆ THỐNG PHÂN TÍCH NGUỒN GỐC VÀ PHÂN LỚP DỮ LIỆU ĐA CHIỀU</h2>
        <div class="status-banner">{status_text}</div>

        <div class="stats-grid">
            <div class="stat-card">
                <div class="stat-title">Khối lượng sự kiện thu thập</div>
                <div class="stat-value" style="color: #2980b9;">{pstats['generated']:,}</div>
            </div>
            <div class="stat-card">
                <div class="stat-title">Dung lượng sự kiện xử lý</div>
                <div class="stat-value" style="color: #27ae60;">{consumed_count:,.0f}</div>
            </div>
            <div class="stat-card">
                <div class="stat-title">Thời gian vận hành (Uptime)</div>
                <div class="stat-value" style="color: #d35400;">{elapsed:.1f}s</div>
            </div>
        </div>

        <div class="report-grid">
            <div class="report-col" style="flex: 1.3;">
                <div class="report-title">Bản Ghi Dữ Liệu Thời Gian Thực (Transaction Logs)</div>
                <table class="data-table">
                    <thead>
                        <tr>
                            <th>Nội dung biến đổi (Payload)</th>
                            <th style="text-align: center;">Phân lớp (Actor Class)</th>
                        </tr>
                    </thead>
                    <tbody>
                        {table_rows if table_rows else '<tr><td colspan="2" style="text-align:center; padding:30px; color: #7f8c8d; font-style: italic;">Hệ thống đang tiến hành thu thập mẫu dữ liệu...</td></tr>'}
                    </tbody>
                </table>
            </div>
            <div class="report-col" style="flex: 1; display: flex; flex-direction: column; align-items: center; justify-content: flex-start;">
                <div class="report-title" style="width: 100%;">Mô Hình Khảo Sát Tần Suất (2D Projection)</div>
                <img src="{chart_img_src}" style="max-width: 100%; height: auto; margin-top: 5px;" alt="Mô hình phân phối dữ liệu 2 chiều"/>
            </div>
        </div>
    </div>
    """
    return html

# ------------------------------------------------------------------------------
# 4. CHU TRÌNH ĐIỀU PHỐI (EXECUTION PIPELINE)
# ------------------------------------------------------------------------------
def run_stream_demo():
    stop_event = threading.Event()
    producer = Producer(PRODUCER_CONF)
    consumer = Consumer(CONSUMER_CONF)

    rows = []
    consumed_count = 0

    try: consumer.subscribe([TOPIC])
    except: pass

    producer_thread = threading.Thread(target=wikimedia_streaming_producer_worker, args=(producer, stop_event), daemon=True)
    producer_thread.start()

    started_at = time.monotonic()
    last_render = 0.0

    dash_handle = display(HTML("<div style='font-family: sans-serif; color: #555;'>Khởi tạo phân hệ giám sát. Vui lòng chờ...</div>"), display_id="live_monitor")

    try:
        while time.monotonic() - started_at < DURATION_SECONDS:
            try:
                message = MockMessage(local_queue.get(timeout=0.05))
            except queue.Empty:
                message = None

            if message is not None and not message.error():
                try:
                    event = json.loads(message.value().decode('utf-8'))
                    if event.get('run_id') == RUN_ID:
                        consumed_count += 1
                        rows.append(event)
                        # Giới hạn kích thước bộ nhớ đệm nhằm tối ưu hóa RAM
                        if len(rows) > 1000: rows.pop(0)
                except: pass

            current_time = time.monotonic()
            if current_time - last_render >= 1.0:
                html_content = generate_full_dashboard(rows, consumed_count, started_at)
                dash_handle.update(HTML(html_content))
                last_render = current_time

    except KeyboardInterrupt: pass
    finally:
        stop_event.set()
        producer_thread.join(timeout=2)
        consumer.close()

    dash_handle.update(HTML(generate_full_dashboard(rows, consumed_count, started_at)))
    print("\nPhiên giám sát luồng dữ liệu thời gian thực đã hoàn tất.")
    return pd.DataFrame(rows)

results_df = run_stream_demo()